In [ ]:
import json
import pandas as pd
import os
from ftplib import FTP
from itertools import chain
import zipfile
import fnmatch
from jsonpath_ng import parse
import shutil
from joblib import Parallel, delayed

L'objectif de ce notebook est d'extraire les informations pertinentes des fichiers du rne qui sont stockés sur un serveur ftp.

## Téléchargement de la base 

Il faut tout d'abord se créer un compte sur le site *inpi* et récupérer son host, user password pour le serveur ftp. On télécharge le fichier stock_rne_formalites

In [ ]:
host = 
user = 
password = 

In [ ]:
fichier = "stock_RNE_formalites_20250523_0000.zip"

In [ ]:
ftp = FTP(host, user, password)

# Télécharger le fichier
with open(fichier, "wb") as f:
    ftp.retrbinary(f"RETR {fichier}", f.write)

# Fermer la connexion
ftp.quit()

## Traitement de la base

On commence par dézipper le fichier 

In [ ]:
#with zipfile.ZipFile(fichier, 'r') as zip_ref:
#    zip_ref.extractall(fichier.split('.')[0])

#pour dézipper uniquement une partie des fichiers avec des codes inférieurs à stock_000100 pour faire des tests
pattern = "stock_000*.*"
with zipfile.ZipFile(fichier, "r") as z:
    for member in z.namelist():
        if fnmatch.fnmatch(member, pattern):
            z.extract(member, fichier.split('.')[0])

On va maintenant boucler sur les fichiers pour extraire les informations sur les dirigeants (y compris les sociétés)

In [ ]:
def extraction_rne(df):

    results = []
    expr_pm = parse('$.formality.content.personneMorale.composition.pouvoirs')
    expr_pf = parse('$.formality.content.personnePhysique.identite.entrepreneur.descriptionPersonne')
    
    cols = ['siren', 'typePersonne', 'nom', 'prenoms', 'genre', 'nomUsage', 'dateDeNaissance', 'roleEntreprise', 'indicateurActifAgricole', 
     'actif', 'entreprise.siren', 'entreprise.roleEntreprise', 
     'individu.descriptionPersonne.nom', 'individu.descriptionPersonne.prenoms', 'individu.descriptionPersonne.genre', 'individu.descriptionPersonne.nomUsage', 
     'individu.descriptionPersonne.dateDeNaissance']
    for ent in df:
            # pour les personneMorale
            liens_pm = list(chain.from_iterable([match.value for match in expr_pm.find(ent)])) 
            for lien in liens_pm:
                lien['siren'] = ent.get("siren")
                lien['typePersonne'] = "personneMorale"
            results.extend(liens_pm)
    
            # pour les personnePhysique
            liens_pf = [{"siren": ent.get("siren"), "typePersonne" : "personnePhysique" ,**match.value} for match in expr_pf.find(ent)]
            results.extend(liens_pf)
    results = pd.json_normalize(results) 
            
    return(results.loc[:, results.columns.intersection(cols)])

In [ ]:
files = os.listdir(fichier.split('.')[0])

##### L'opération est très longue, il vaut mieux paralléliser les calculs

In [ ]:
%%time
results = []
for file in files:
    with open(fichier.split('.')[0] + '/' + file) as f:
        data = json.load(f)
    results.append(extraction_rne(data))

In [ ]:
rne = pd.concat(results)

In [ ]:
rne.to_parquet(fichier.split('.')[0] + '.parquet' )

##### L'opération est très longue, on peut donc parallélisier les calculs

In [ ]:
def calcul_parallel(file):
    with open(fichier.split('.')[0] + '/' + file) as f:
        data = json.load(f)
    return(extraction_rne(data))

In [ ]:
%%time
results = Parallel(n_jobs=200)(delayed(calcul_parallel)(file) for file in files)

In [ ]:
rne = pd.concat(results)

In [ ]:
rne

In [ ]:
rne.to_parquet(fichier.split('.')[0] + '.parquet' )

# Attention, cela prend trop de place sur onyxia, il faut dézipper en plusieurs fois

In [ ]:
temp_dir = "temp_extract2"
batch_size = 500

def calcul_parallel(file):
    with open(temp_dir + '/' + file) as f:
        data = json.load(f)
    return(extraction_rne(data))

def process_file(temp_dir):
    files = os.listdir(temp_dir)
    return(Parallel(n_jobs=100)(delayed(calcul_parallel)(file) for file in files))


with zipfile.ZipFile(fichier, 'r') as z:
    file_list = z.namelist()
    print("Number of files:", len(file_list))

results = []
with zipfile.ZipFile(fichier, 'r') as z:
    members = z.infolist()

    for i in range(0, len(members), batch_size):
        batch = members[i:i + batch_size]

        print(f"\nProcessing batch {i//batch_size + 1}")

        # Extract batch
        for member in batch:
            z.extract(member, temp_dir)
        
        results.append(process_file(temp_dir))
        shutil.rmtree(temp_dir)


In [ ]:
flat = list(chain.from_iterable(results))
rne = pd.concat(flat)

In [ ]:
rne[rne.siren=="353192263"]

In [ ]:
rne[~rne['individu.descriptionPersonne.genre'].isna()]

In [ ]:
rne.to_parquet(fichier.split('.')[0] + '.parquet', index=False) 

#### Autres

In [ ]:
rne.count()

In [ ]:
shutil.rmtree(temp_dir)

In [ ]:
shutil.rmtree(fichier.split('.')[0])

In [ ]:
!mc ls s3/<ton_bucket>